### Extracted and Loaded Data Saved in Google Big Query 
4 table were created earlier to store the content extracted from 'YFinance' and 'Fred'. The `sources.yml` file store the list of the tables.

In [3]:
%%bash 
cat stock_analysis/models/sources.yml

version: 2

sources:
  - name: stock_analysis
    database: ntuproj-462609
    tables:
      - name: combined_tickers
      - name: fred
      - name: stock_info
      - name: techincal_indicator


1. `combined_tickers` table consolidates the data from YFinance. Daily opening, closing, highest, lowest prices, volume traded, dividend and stock splits are found here.
  
    ![](assets/combined_tickers.png)

2. `fred` table records macroeconomic data such as inflation rate, interest rate, unemployment rates, GDP, 10 years treasury yield, industry production and consumer sentiment.

    ![](assets/fred.png) 

3.  `stock_info` table contains company information, including name, description, country, sector, market capitalization (market cap), price-to-earnings ratio (PE), and earnings per share (EPS). 

    ![](assets/fred.png) 

4. `technical_indicators` table stores technical analysis indicators like simple moving average (SMA), exponential moving average (EMA), and others.

    ![](assets/fred.png)

### Setup DBT

Initialize a new dbt project named `stock_analysis` and configure the dbt project through the interactive terminal. 
(Note: Only run the bash command in Ubuntu CLI, not Jupyter Notebook. Jupyter Notebook cannot support interative terminal)

```bash
dbt init stock_analysis

### Interactive Terminal
Which database would you like to use?
[1] bigquery
Enter a number: 1

[1] oauth
[2] service_account
Desired authentication method option (enter a number): 1

project (GCP project id): <your_project_id>

dataset (the name of your dbt dataset): yfinance_econdata

threads (1 or more): 1

job_execution_timeout_seconds [300]:

[1] US
[2] EU
Desired location option (enter a number): 1
```

### Check DBT profile
Review `stock_analysis/profiles.yml` to ensure the profile information is added correctly. 

In [7]:
%%bash
cat stock_analysis/profiles.yml

stock_analysis:
  target: dev
  outputs:
    threads: 1
    location: US
    priority: interactive
    dev:
      type: bigquery
      method: oauth
      project: ntuproj-462609
      dataset: stock_analysis
      retries: 2


### DBT project directory - stock_analysis 
We use `tree` bash command to analyse the `stock_analysis` dbt project structure.
To run `tree`, please ensure that `tree` is installed in advanced. 
```bash
sudo apt install tree 
```
Check `stock_analysis` dbt project is created. Change from the working directory to the newly created `stock_analysis` dbt project folder.

In [13]:
%%bash
cd stock_analysis
tree -L 2

.
├── README.md
├── analyses
├── dbt_packages
│   ├── dbt_date
│   ├── dbt_expectations
│   └── dbt_utils
├── dbt_project.yml
├── logs
│   └── dbt.log
├── macros
├── models
│   ├── dim_date.sql
│ �� ├── dim_info.sql
│   ├── fact_security.sql
│   ├── schema.yml
│   ��── schema.yml:Zone.Identifier
│   └── sources.yml
├── package-lock.yml
├─�� packages.yml
├── profiles.yml
├── schema.yml
├── seeds
├── snapshots
├── target
│   ├── compiled
│   ├��─ graph.gpickle
│   ├── graph_summary.json
│   ├── manifest.json
│   ├���─ partial_parse.msgpack
│   ├── run
│   ├── run_results.json
│   └── semantic_manifest.json
└── tests

15 directories, 19 files


### Target FACT and DIM Table Schema  

The `schema.yml` file defines the structure of the `fact_security`, `dim_date`, and `dim_info` tables. It serves as the foundation for running DBT tests to ensure data integrity and compliance with the specified schema. For data validation, the project utilizes packages such as `dbt_utils` and `dbt_expectations`, which provide robust testing capabilities and validation functions.  


In [ ]:
%%bash
cat stock_analysis/packages.yml

packages:
  - package: dbt-labs/dbt_utils
    version: 1.1.1
  - package: calogica/dbt_expectations
    version: 0.10.4

In [12]:
%%bash
cat stock_analysis/models/schema.yml

version: 2

models:
  - name: fact_security
    description: "Fact table for security"
    tests:
      - dbt_expectations.expect_compound_columns_to_be_unique:
          column_list: ["ticker_symbol", "date"]
    columns:
      - name: date
        description: "The foreign key to the date dimension"
        tests:
          - not_null
          - dbt_utils.accepted_range:
              max_value: "CURRENT_DATE()"
          - relationships:
              to: ref('dim_date')
              field: date
          - dbt_expectations.expect_column_values_to_be_of_type:
              column_type: date
      - name: ticker_symbol
        description: "The foreign key to the info dimension"
        tests:
          - not_null
          - relationships:
              to: ref('dim_info')
              field: ticker_symbol
          - dbt_expectations.expect_column_values_to_be_of_type:
              column_type: string
      - name: close_price
        description: "close price of the security"


1. `fact_security` table contains stock daily information, including ticker data (open, close, high, and low prices, volume traded), corporate actions such as dividends and stock splits, technical indicators, EPS and PE ratios, and relevant economic data. 

In [4]:
%%bash
cat stock_analysis/models/fact_security.sql

{{
    config(
        materialized='table'
    )
}}

WITH dup_data AS (
SELECT
  t.ticker_symbol,
  DATE(t.date) AS date,
  t.close_price,
  t.high_price,
  t.low_price,
  t.open_price,
  t.stock_splits,
  t.dividend,
  t.volume_traded,
  f.GDP_growth_rate,
  f.UMCSENT,
  f.inflation_rate,
  f.interest_rate,
  f.unemployment_rate,
  f.industrial_production,
  f.10_yr_tresaury,
  i.forward_eps,
  i.forward_pe,
  i.market_cap,
  i.trailing_eps,
  i.trailing_pe,
  ta.ema_50,
  ta.rsi_14,
  ta.sma_50
FROM {{source('stock_analysis', 'combined_tickers')}} t
LEFT JOIN {{source('stock_analysis', 'fred')}} f
  ON DATE(t.date) = DATE(f.date)
LEFT JOIN {{source('stock_analysis', 'stock_info')}} i
  ON t.ticker_symbol = i.ticker_symbol AND DATE(t.date) BETWEEN '2025-04-01' AND '2025-06-30'
LEFT JOIN {{source('stock_analysis','techincal_indicator')}} ta
  ON DATE(t.date) = DATE(ta.date) and t.ticker_symbol = ta.ticker_symbol
)
SELECT 
  ticker_symbol,
  date,
  ANY_VALUE(close_price) AS close_pric


2. `dim_info` table stores the stock-related information, primarily focusing on company name, description, country, sector, and industry.

In [10]:
%%bash
cat stock_analysis/models/dim_info.sql

{{
    config(
        materialized='table'
    )
}}

SELECT
  ticker_symbol,
  company_description,
  company_name,
  country,
  industry,
  sector
FROM {{source('stock_analysis', 'stock_info')}}


3. `dim_date` table provides a detailed breakdown of dates to support easy reference and enhance analytics processing.


In [11]:
%%bash
cat stock_analysis/models/dim_date.sql

{{ 
    config(
        materialized='table'
    )
}}

WITH deduplicated_dates AS (
    SELECT DISTINCT date
    FROM {{ source('stock_analysis', 'combined_tickers') }}
)
SELECT
    DATE(date) AS date,
    EXTRACT(YEAR FROM date) AS year,
    EXTRACT(MONTH FROM date) AS month,
    EXTRACT(DAY FROM date) AS day,
    EXTRACT(QUARTER FROM date) AS quarter,
    FORMAT_TIMESTAMP('%A', date) AS day_of_week
FROM 
    deduplicated_dates


### Target FACT and DIM Table Entity Relationship Diagram
![security ERD](assets/security_erd.svg)

### Create and Validate FACT and DIM Tables with DBT

1. The `dbt deps` command is  install and manage dependencies (`dbt_expectations` and `dbt_utils`) specified in the packages.yml file of a dbt project.

In [1]:
%%bash
cd stock_analysis
dbt deps

15:31:58  Running with dbt=1.9.2
15:31:59  [WARNING]: Deprecated functionality
The `calogica/dbt_expectations` package is deprecated in favor of
`metaplane/dbt_expectations`. Please update your `packages.yml` configuration to
use `metaplane/dbt_expectations` instead.
15:31:59  Installing dbt-labs/dbt_utils
15:32:00  Installed from version 1.1.1
15:32:00  Updated version available: 1.3.0
15:32:00  Installing calogica/dbt_expectations
15:32:03  Installed from version 0.10.4
15:32:03  Up to date!
15:32:03  Installing calogica/dbt_date
15:32:03  Installed from version 0.10.1
15:32:03  Up to date!
15:32:03  
15:32:03  Updates available for packages: ['dbt-labs/dbt_utils']                 
Update your versions in packages.yml, then run dbt deps


2. The `dbt run` command executes the SQL models (`fact_securities`, `dim_info`, `dim_date`) defined in a dbt project and builds the corresponding tables in the target dataset `stock_analysis`. 

In [9]:
%%bash
cd stock_analysis
dbt run

05:17:20  Running with dbt=1.9.2
05:17:25  Registered adapter: bigquery=1.9.1
05:17:28  Found 3 models, 32 data tests, 4 sources, 872 macros
05:17:28  
05:17:28  Concurrency: 1 threads (target='dev')
05:17:28  
05:17:34  1 of 3 START sql table model stock_analysis.dim_date ........................... [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:17:41  1 of 3 OK created sql table model stock_analysis.dim_date ...................... [CREATE TABLE (1.5k rows, 176.8 KiB processed) in 6.74s]
05:17:41  2 of 3 START sql table model stock_analysis.dim_info ........................... [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:17:47  2 of 3 OK created sql table model stock_analysis.dim_info ...................... [CREATE TABLE (15.0 rows, 11.9 KiB processed) in 5.84s]
05:17:47  3 of 3 START sql table model stock_analysis.fact_security ...................... [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:17:54  3 of 3 OK created sql table model stock_analysis.fact_security ................. [CREATE TABLE (22.6k rows, 3.8 MiB processed) in 7.38s]
05:17:54  
05:17:54  Finished running 3 table models in 0 hours 0 minutes and 26.84 seconds (26.84s).
05:17:55  
05:17:55  Completed successfully
05:17:55  
05:17:55  Done. PASS=3 WARN=0 ERROR=0 SKIP=0 TOTAL=3


3. The `dbt test` command runs tests defined in a dbt project to validate data quality and integrity. It includes both built-in tests, such as checking for unique values or non-null fields, and custom tests defined in SQL. 

In [11]:
%%bash
cd stock_analysis
dbt test

05:21:34  Running with dbt=1.9.2
05:21:39  Registered adapter: bigquery=1.9.1
05:21:45  Found 3 models, 29 data tests, 4 sources, 872 macros
05:21:45  
05:21:45  Concurrency: 1 threads (target='dev')
05:21:45  
05:21:50  1 of 29 START test dbt_expectations_expect_column_values_to_be_of_type_dim_date_date__date  [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:21:55  1 of 29 PASS dbt_expectations_expect_column_values_to_be_of_type_dim_date_date__date  [PASS in 4.60s]
05:21:55  2 of 29 START test dbt_expectations_expect_column_values_to_be_of_type_dim_info_company_description__string  [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:22:00  2 of 29 PASS dbt_expectations_expect_column_values_to_be_of_type_dim_info_company_description__string  [PASS in 5.22s]
05:22:00  3 of 29 START test dbt_expectations_expect_column_values_to_be_of_type_dim_info_company_name__string  [RUN]


...

05:23:29  27 of 29 PASS unique_dim_info_company_description .............................. [PASS in 2.50s]
05:23:29  28 of 29 START test unique_dim_info_company_name ............................... [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:23:33  28 of 29 PASS unique_dim_info_company_name ..................................... [PASS in 3.65s]
05:23:33  29 of 29 START test unique_dim_info_ticker_symbol .............................. [RUN]


/home/joshlai/miniconda3/envs/NTUProj/lib/python3.10/site-packages/dbt/adapters/bigquery/connections.py:570: FutureWarning: job_retry must be explicitly set to None if job_id is set.
BigQuery cannot retry a failed job by using the exact
same ID. Setting job_id without explicitly disabling
job_retry will raise an error in the future. To avoid this
warning, either use job_id_prefix instead (preferred) or
set job_retry=None.
  query_job = client.query(


05:23:37  29 of 29 PASS unique_dim_info_ticker_symbol .................................... [PASS in 3.65s]
05:23:37  
05:23:37  Finished running 29 data tests in 0 hours 1 minutes and 51.82 seconds (111.82s).
05:23:37  
05:23:37  Completed successfully
05:23:37  
05:23:37  Done. PASS=29 WARN=0 ERROR=0 SKIP=0 TOTAL=29


# Limitations of the Current DBT Implementation

| **Category**                | **Description**                                                                                                     | **Potential Solution**                                                                                      |
|------------------------------|---------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------|
| **Data Type Validation**     | Validation of `float` and `integer` types using `dbt_expectations.expect_column_values_to_be_of_type` failed. These tests were commented out. | Investigate root causes, update DBT expectations package, or adjust source schema definitions.             |
| **ETF-Specific Test Cases**  | ETF tickers lack country, sector, and industry metadata, unlike stocks. Some `not_null` checks were commented out. | Develop separate schema and validation logic for ETF-specific columns.                                     |
| **Regex Validation**         | No regular expressions were used to validate column patterns (e.g., ticker symbols, ISO country codes).           | Add regex-based tests to enforce column-specific format rules.                                             |
| **Range Validation**         | Did not check max/min values for numeric columns (e.g., stock prices, volume).                                    | Implement boundary checks for numeric data to ensure values fall within realistic thresholds.              |
| **Set-Based Validation**     | Tests to validate column values against an expected list (e.g., valid industry codes) were not implemented.        | Use `dbt_expectations.expect_column_values_to_be_in_set` to enforce compliance with predefined reference data. |
| **General Time Constraints** | Some additional validations (e.g., edge cases, hierarchical relationships) were not included due to time limits.   | Extend test coverage to include edge cases and domain-specific logic in future iterations.                 |
